In [1]:
import subprocess

CHECK = "\u2705"
FAIL  = "\u274C"
WARN  = "\u26A0"
# --------------------------------------------------------------------
def git_clone(user, repo, folders, files):
    
    url      = f'https://github.com/{user}/{repo}'
    raw_url  = url.replace('github.com', 'raw.githubusercontent.com')
    raw_url += '/refs/heads/main/'

    # 1. Delete local repo folder if it exists
    result = subprocess.run(
        ["rm", "-rf", repo],
        capture_output=True)
    if result.returncode != 0:
        print(f"{FAIL} Deletion of {repo} failed:",
              result.stderr.strip())
        return result.returncode

    # 2. Clone repo <repo> with minimal history
    result = subprocess.run(
        ["git", "clone",
         "--depth", "1",
         "--filter=blob:none",
         "--sparse", url],
        capture_output=True,
        text=True)
    if result.returncode == 0:
        print(result.stdout.strip())
    else:
        print(f"{FAIL} Clone of {repo} failed:", result.stderr.strip())
        return result.returncode
    
    # 3. Checkout specified folders
    os.chdir(repo)
    for dirname in folders:
    	result = subprocess.run(
        	["git", "sparse-checkout",
          	"set", dirname],
        	capture_output=True,
        	text=True)
    	if result.returncode == 0:
        	print(result.stdout.strip())
    	else:
        	print(f"{FAIL} Sparse checkout of {dirname} failed:", 
			result.stderr.strip())
        	return result.returncode
        
    for file in files:
        url = raw_url + file
        result = subprocess.run(
            ["wget", "-q", url, "-O", file],
            capture_output=True, text=True)
        if result.returncode != 0:
            print(f"{FAIL} wget of {file} failed:", result.stderr.strip())
            return result.returncode
            
    # 4. Remember to go back to code folder
    os.chdir("..")
 
    print(f"{CHECK} Clone of '{repo}' successful\n")

In [ ]:
try:
    # -----------------------------------------------------
    # Clone from a GitHub repo
    # -----------------------------------------------------
    from google.colab import drive
    drive.mount('/content/gdrive')
    print('\nGoogle Drive mounted\n')
    %cd /content/gdrive/MyDrive/{COLAB_FOLDER}
    pwd = %pwd 
    print(pwd)
    print()
    
    print('git clone to Colab')
    print(f' GitHub(user):    {GITHUB_USER}')
    print(f' GitHub(repo):    {GITHUB_REPO}')
    print(f' GitHub(folders): {GITHUB_FOLDERS}')
    print(f' GitHub(files):   {GITHUB_FILES}')
    print()
    
    %rm -rf {GITHUB_REPO}
    %rm -f  bootstrap.py
    %ls
    github = 'raw.githubusercontent.com'
    stub   = 'refs/heads/main'
    !wget -q https://{github}/{GITHUB_USER}/{GITHUB_REPO}/{stub}/bootstrap.py
    print()
    
    # Download the pieces of the repo we need
    from bootstrap import git_clone
    git_clone(user=GITHUB_USER, 
              repo=GITHUB_REPO, 
              folders=GITHUB_FOLDERS, 
              files=GITHUB_FILES)
    %ls
    print()
    
    print('Running on Colab')
    IN_COLAB = True
except:
    print('Running locally')
    IN_COLAB = False